# Wan2GP on Kaggle

Sets up [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) in a fresh GPU-backed Kaggle notebook session. 

Run the cells in order to prepare the runtime, install dependencies, and launch the Gradio interface. Click on the link in the output from the last cell to launch the app in your browser.

> **Kaggle VRAM note:** Kaggle commonly assigns 16 GB GPUs such as T4 or P100. Most Wan2GP models exceed that budget; the Wan 2.2 TextImage2Video FastWan model is the best starting point.


## 1. Confirm the accelerator

Open the notebook settings panel and select **GPU** as the accelerator before running anything else. Also enable **Internet** so Kaggle can clone Wan2GP and install packages.

If this cell raises an error, go back to the notebook settings, pick **GPU**, save, and rerun this cell.


In [ ]:
import subprocess

try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'GPU not detected. In Kaggle notebook settings, select GPU T4 as the accelerator when available, or GPU P100 as a fallback, save, then rerun this cell. If GPU options are grayed out, make sure your Kaggle account has completed phone verification.'
    ) from exc


## 2. Configure the workspace path and data storage


Kaggle's `/kaggle/working` directory is the notebook output area and has a smaller quota than the full session disk. This notebook stores large checkpoints, LoRAs and model caches in `/kaggle/temp`, while generated outputs stay in `/kaggle/working/Wan2GP-outputs` so they can be saved with the notebook version.


In [ ]:
from pathlib import Path


KAGGLE_WORKING_ROOT = Path('/kaggle/working').resolve()
KAGGLE_TEMP_ROOT = Path('/kaggle/temp').resolve()
WAN2GP_ROOT = (KAGGLE_WORKING_ROOT / 'Wan2GP').resolve()
WAN_SESSION_DATA_ROOT = (KAGGLE_TEMP_ROOT / 'Wan2GP-data').resolve()
WAN_OUTPUTS_DIR = (KAGGLE_WORKING_ROOT / 'Wan2GP-outputs').resolve()
data_mode = 'Kaggle temporary session storage for models; Kaggle working output storage for generated files'

WAN_CKPTS_DIR = (WAN_SESSION_DATA_ROOT / 'ckpts').resolve()
WAN_LORAS_DIR = (WAN_SESSION_DATA_ROOT / 'loras').resolve()
WAN_CACHE_DIR = (WAN_SESSION_DATA_ROOT / 'cache').resolve()
WAN_LTX2_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2').resolve()
WAN_LTX2_22B_LORAS_DIR = (WAN_LORAS_DIR / 'ltx2_22B').resolve()

WAN2GP_ROOT.parent.mkdir(parents=True, exist_ok=True)
WAN_SESSION_DATA_ROOT.mkdir(parents=True, exist_ok=True)
WAN_CKPTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
WAN_CACHE_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_LORAS_DIR.mkdir(parents=True, exist_ok=True)
WAN_LTX2_22B_LORAS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Wan2GP repository path: {WAN2GP_ROOT}')
print(f'Data storage mode: {data_mode}')
print(f'Session data root: {WAN_SESSION_DATA_ROOT}')
print(f'Checkpoints: {WAN_CKPTS_DIR}')
print(f'LoRAs: {WAN_LORAS_DIR}')
print(f'LTX-2 LoRAs: {WAN_LTX2_LORAS_DIR}')
print(f'LTX-2 22B LoRAs: {WAN_LTX2_22B_LORAS_DIR}')
print(f'Outputs: {WAN_OUTPUTS_DIR}')
print(f'Cache: {WAN_CACHE_DIR}')


## 3. Download or update Wan2GP

Clone the repository if it is not present yet; otherwise pull the latest changes.


In [ ]:
import shutil, subprocess
from pathlib import Path

def merge_directory_contents(source_dir: Path, destination_dir: Path) -> None:
    for child in list(source_dir.iterdir()):
        destination = destination_dir / child.name
        if destination.exists():
            if child.is_dir() and destination.is_dir():
                merge_directory_contents(child, destination)
                child.rmdir()
                continue
            raise RuntimeError(f'Cannot move {child} into {destination_dir}: {destination} already exists.')
        shutil.move(str(child), str(destination))

def attach_data_directory(repo_path: Path, data_path: Path) -> None:
    data_path.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        if repo_path.resolve() != data_path.resolve():
            raise RuntimeError(f'{repo_path} already points to {repo_path.resolve()}, expected {data_path}.')
        print(f'Using existing link: {repo_path} -> {data_path}')
        return

    if repo_path.exists():
        if not repo_path.is_dir():
            raise RuntimeError(f'Expected a directory at {repo_path}.')
        merge_directory_contents(repo_path, data_path)
        repo_path.rmdir()
    else:
        repo_path.parent.mkdir(parents=True, exist_ok=True)

    repo_path.symlink_to(data_path, target_is_directory=True)
    print(f'Linked {repo_path.name} -> {data_path}')

repo_url = 'https://github.com/deepbeepmeep/Wan2GP.git'
if WAN2GP_ROOT.exists():
    status = subprocess.run(
        ['git', '-C', str(WAN2GP_ROOT), 'status', '--porcelain'],
        check=True,
        capture_output=True,
        text=True,
    )
    if status.stdout.strip():
        print('Repository already exists and has local changes. Skipping git pull to preserve your persistent files.')
    else:
        print('Repository already exists. Pulling latest changes...')
        subprocess.run(['git', '-C', str(WAN2GP_ROOT), 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(WAN2GP_ROOT)], check=True)

attach_data_directory(WAN2GP_ROOT / 'ckpts', WAN_CKPTS_DIR)
attach_data_directory(WAN2GP_ROOT / 'loras', WAN_LORAS_DIR)
attach_data_directory(WAN2GP_ROOT / 'outputs', WAN_OUTPUTS_DIR)


## 4. Install system dependencies

Install shared libraries needed for video and audio processing. If you see a warning about skipping an extra repository, it is safe to ignore.

In [ ]:
import os, shutil, subprocess

env = os.environ.copy()
env['DEBIAN_FRONTEND'] = 'noninteractive'

apt_prefix = [] if getattr(os, 'geteuid', lambda: 0)() == 0 else ['sudo'] if shutil.which('sudo') else []

subprocess.run(apt_prefix + ['apt-get', 'update', '-qq'], check=True, env=env)
subprocess.run([
    *apt_prefix, 'apt-get', 'install', '-y', '--no-install-recommends',
    'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'
], check=True, env=env)


## 5. Install Python dependencies

Use Kaggle's preinstalled CUDA-enabled PyTorch stack and install only Wan2GP's missing Python packages. Wan2GP will use its default attention mode.


In [ ]:
import importlib, os, subprocess, sys

env = os.environ.copy()
env.setdefault('DEBIAN_FRONTEND', 'noninteractive')
env['PIP_NO_CACHE_DIR'] = '1'
env['PIP_CACHE_DIR'] = str(WAN_CACHE_DIR / 'pip')
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle PyTorch is installed, but CUDA is not available. Check that the Kaggle accelerator is set to GPU, then restart the session and rerun this notebook.')

installed = {'torch': torch.__version__}
for name in ('torchvision', 'torchaudio'):
    try:
        module = importlib.import_module(name)
        installed[name] = module.__version__
    except Exception:
        installed[name] = None

constraints = WAN_CACHE_DIR / 'kaggle-torch-constraints.txt'
constraint_lines = [f'torch=={torch.__version__.split("+", 1)[0]}']
for name in ('torchvision', 'torchaudio'):
    version = installed.get(name)
    if version:
        constraint_lines.append(f'{name}=={version.split("+", 1)[0]}')
constraints.write_text('\n'.join(constraint_lines) + '\n')

print('Using Kaggle PyTorch stack:', ', '.join(f'{name} {version}' for name, version in installed.items() if version))
print(f'CUDA available: {torch.cuda.get_device_name(0)} / CUDA {torch.version.cuda}')
print('Installing Wan2GP requirements while keeping Kaggle PyTorch pinned...')

subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', 'setuptools', 'wheel'], check=True, env=env)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '--no-cache-dir',
    '--upgrade-strategy',
    'only-if-needed',
    '-r',
    str(WAN2GP_ROOT / 'requirements.txt'),
    '-c',
    str(constraints),
], check=True, env=env)
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], check=False, env=env)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch CUDA became unavailable after installing requirements. Restart the Kaggle session and rerun Step 5.')

print(f'Final torch: {torch.__version__} / CUDA {torch.version.cuda}')
print('Wan2GP dependencies installed.')


## 6. Force a headless matplotlib backend

Ensure Wan2GP's preprocessing tools use the headless Agg backend so Step 7 launches cleanly in Kaggle.


In [ ]:
from pathlib import Path

# Replace the TkAgg backend with the headless Agg backend if present.
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
needle = "matplotlib.use('TkAgg')"
replacement = "matplotlib.use('Agg')"

if not target.exists():
    print(f'Skipping: {target} not found.')
else:
    text = target.read_text()
    if replacement in text:
        print('Agg backend already set; no change needed.')
    elif needle in text:
        target.write_text(text.replace(needle, replacement, 1))
        print('Replaced TkAgg with Agg in interact_tools.py.')
    else:
        print('Backend call not found; no change made.')


## 7. Launch Wan2GP

Run the Gradio interface. You will find the gradio link in the output. Click on the link to access the UI. Keep the cell running to stay connected; stop it with the square **Stop** button when you are finished.

In [ ]:
import os, subprocess, sys, threading, time

env = os.environ.copy()
env['WAN_CACHE_DIR'] = str(WAN_CACHE_DIR)
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')
env['HUGGINGFACE_HUB_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'hub')
env['TRANSFORMERS_CACHE'] = str(WAN_CACHE_DIR / 'huggingface' / 'transformers')
env['TORCH_HOME'] = str(WAN_CACHE_DIR / 'torch')
env['XDG_CACHE_HOME'] = str(WAN_CACHE_DIR / '.cache')
cmd = [
    sys.executable,
    '-u',
    'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--share',
    '--profile', '5',
]

print('Using Kaggle temporary session storage for checkpoints, LoRAs and caches.')
print(f'Generated outputs will be written to {WAN_OUTPUTS_DIR}.')
print('Launching Wan2GP…')
process = subprocess.Popen(
    cmd,
    cwd=str(WAN2GP_ROOT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
stop_event = threading.Event()

def keepalive():
    while not stop_event.is_set():
        time.sleep(45)
        if stop_event.is_set():
            break
        print('[keepalive] Notebook cell still running…')

keepalive_thread = threading.Thread(target=keepalive, daemon=True)
keepalive_thread.start()

try:
    for line in iter(process.stdout.readline, ''):
        if not line:
            break
        print(line, end='')
except KeyboardInterrupt:
    print('Stopping Wan2GP…')
    process.terminate()
finally:
    stop_event.set()
    process.wait()
    keepalive_thread.join(timeout=1)
    print(f'Wan2GP stopped (return code: {process.returncode}).')
